# Notebook 21: Reproducibility & Robustness Validation

**Date**: 2026-01-12  
**Goal**: Validate stability of results with multi-seed runs and larger samples

**Motivation**: NB20 showed 60 percentage point swings in spline advantage for same region:  
- Himalayas L=10: -8.83% to +52.0%  
- Sahara L=10: +1.96% to +13.37%

**This notebook establishes reproducible baselines before making any claims.**

---

## Experiments

### Experiment 1: Global Multi-Seed Validation (NB19 Reproducibility)
- **Config**: 15K samples, global, L=10, 3 activations
- **Seeds**: 10 runs (seeds 42-51)
- **Question**: Are NB19's +0.36% spline advantage reproducible?
- **Expected**: Low variance (global scale, large sample)

### Experiment 2: Sample Size Sensitivity
- **Sample sizes**: 5K, 10K, 20K, 50K (global)
- **Seeds**: 10 runs each
- **Question**: At what sample size do results stabilize?
- **Expected**: Variance decreases with sample size

### Experiment 3: Regional Multi-Seed (Larger Samples)
- **Regions**: Himalayas (mountain), Sahara (flat)
- **Sample sizes**: 10K, 20K per region
- **Seeds**: 10 runs each
- **Question**: Do terrain patterns stabilize with more data?
- **Expected**: If real effect, mean should be consistent with low std

### Experiment 4: Extended Training (Convergence)
- **Epochs**: 200 (vs 100 in NB19/20)
- **Seeds**: 5 runs
- **Question**: Were models undertrained in NB19/20?
- **Expected**: Plateaus should emerge, validate 100 epochs was enough

### Experiment 5: Spatial Blocking Sensitivity
- **Grid sizes**: 1°, 2°, 5°, 10° spatial blocks
- **Seeds**: 5 runs each
- **Question**: How sensitive are results to blocking strategy?
- **Expected**: Smaller blocks (more test cells) → lower variance

---

## Success Criteria

**For results to be trustworthy:**
1. Standard deviation < 2% of mean across 10 seeds
2. Spline advantage confidence interval excludes zero (if claiming advantage)
3. Results consistent across sample sizes ≥20K

**If variance remains high:**
- Abandon regional analysis
- Focus on global analysis with error bars
- Publication: "High Variance in Learned Activations for Geographic Data"

---

## Computational Budget

**Exp 1**: 10 seeds × 3 acts × 90s = ~45 min  
**Exp 2**: 4 sizes × 10 seeds × 3 acts × 60-120s = ~4 hours  
**Exp 3**: 2 regions × 2 sizes × 10 seeds × 3 acts × 100s = ~3 hours  
**Exp 4**: 5 seeds × 3 acts × 180s (200 epochs) = ~45 min  
**Exp 5**: 4 grids × 5 seeds × 3 acts × 90s = ~90 min

**Total**: ~10 hours (acceptable for reproducibility validation)

---

## Output Strategy

All results printed in notebook with:
- Mean ± Std for each configuration
- Min/Max values
- Coefficient of variation (std/mean)
- 95% confidence intervals

Optional: Save to Google Drive for persistence

---
## Setup

In [ ]:
# Environment setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

import positional_encoding as PE
import xarray as xr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Model Definitions

In [ ]:
class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class SirenLayer(nn.Module):
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class UniversalEncoder(nn.Module):
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:
            x = self.posenc(coords)

        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


print("✅ Model classes loaded")

---
## Training Utilities

In [ ]:
def sample_global_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample globally with spatial blocking (same as NB19).
    """
    np.random.seed(seed)
    
    # Valid data mask
    valid = data > -1e30
    
    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    # Flatten
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = data[valid]
    
    # Sample subset
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals
    
    # Spatial blocking
    n_lon_cells = int(360 / grid_size)
    n_lat_cells = int(180 / grid_size)
    n_cells = n_lon_cells * n_lat_cells
    
    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))
    
    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon + 180) / grid_size)
        lat_cell = int((lat + 90) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)
    
    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)
    
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def sample_regional_blocked(data, lons, lats, region_bounds, n_samples=5000,
                           grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample from a regional subset with spatial blocking.
    region_bounds: dict with 'lat_min', 'lat_max', 'lon_min', 'lon_max'
    """
    np.random.seed(seed)
    
    # Find indices for region
    lat_mask = (lats >= region_bounds['lat_min']) & (lats <= region_bounds['lat_max'])
    lon_mask = (lons >= region_bounds['lon_min']) & (lons <= region_bounds['lon_max'])
    
    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]
    
    regional_data = data[np.ix_(lat_idx, lon_idx)]
    regional_lats = lats[lat_idx]
    regional_lons = lons[lon_idx]
    
    lon_grid, lat_grid = np.meshgrid(regional_lons, regional_lats)
    valid = regional_data > -1e30
    
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = regional_data[valid]
    
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals
    
    region_width = region_bounds['lon_max'] - region_bounds['lon_min']
    region_height = region_bounds['lat_max'] - region_bounds['lat_min']
    
    n_lon_cells = max(1, int(region_width / grid_size))
    n_lat_cells = max(1, int(region_height / grid_size))
    n_cells = n_lon_cells * n_lat_cells
    
    test_cells = set(np.random.choice(n_cells, max(1, int(n_cells * test_ratio)), replace=False))
    
    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon - region_bounds['lon_min']) / grid_size)
        lat_cell = int((lat - region_bounds['lat_min']) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)
    
    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)
    
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def train_elevation_model(name, encoder, coords_train, vals_train, coords_test, vals_test,
                         epochs=100, batch_size=256, lr=1e-3, verbose=False, track_epochs=False):
    """
    Train elevation prediction model.
    
    If track_epochs=True, returns R² history for convergence analysis.
    """
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    # Normalize
    train_mean, train_std = vals_train.mean(), vals_train.std()
    vals_train_norm = (vals_train - train_mean) / train_std
    vals_test_norm = (vals_test - train_mean) / train_std
    
    shift = -vals_train_norm.min() + 1
    vals_train_shifted = vals_train_norm + shift
    vals_test_shifted = vals_test_norm + shift
    
    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train_shifted), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test_shifted), dtype=torch.float32)
    
    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)
    
    best_r2 = -float('inf')
    r2_history = [] if track_epochs else None
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()
        
        # Evaluate every 10 epochs or at end
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1 or track_epochs:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
            
            if track_epochs:
                r2_history.append(r2)
            
            if verbose and (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")
    
    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    result = {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
    }
    
    if track_epochs:
        result['r2_history'] = r2_history
    
    return result


print("✅ Training utilities loaded")

---
## Statistics Utilities

In [ ]:
def compute_stats(values):
    """
    Compute comprehensive statistics for a list of values.
    """
    values = np.array(values)
    n = len(values)
    mean = values.mean()
    std = values.std(ddof=1) if n > 1 else 0.0
    stderr = std / np.sqrt(n) if n > 1 else 0.0
    
    # 95% confidence interval
    if n > 1:
        ci = stats.t.interval(0.95, n-1, loc=mean, scale=stderr)
    else:
        ci = (mean, mean)
    
    return {
        'n': n,
        'mean': mean,
        'std': std,
        'stderr': stderr,
        'min': values.min(),
        'max': values.max(),
        'ci_low': ci[0],
        'ci_high': ci[1],
        'cv': (std / mean * 100) if mean != 0 else 0.0,  # Coefficient of variation (%)
    }


def print_stats(label, stats_dict):
    """
    Pretty print statistics.
    """
    print(f"{label}:")
    print(f"  Mean ± Std:  {stats_dict['mean']:.4f} ± {stats_dict['std']:.4f}")
    print(f"  Range:       [{stats_dict['min']:.4f}, {stats_dict['max']:.4f}]")
    print(f"  95% CI:      [{stats_dict['ci_low']:.4f}, {stats_dict['ci_high']:.4f}]")
    print(f"  CV:          {stats_dict['cv']:.2f}%")
    print(f"  N:           {stats_dict['n']}")


def compare_activations(relu_r2s, spline_r2s):
    """
    Compare ReLU vs Spline with statistical tests.
    """
    relu_r2s = np.array(relu_r2s)
    spline_r2s = np.array(spline_r2s)
    
    # Paired t-test (same seeds)
    if len(relu_r2s) == len(spline_r2s) and len(relu_r2s) > 1:
        t_stat, p_value = stats.ttest_rel(spline_r2s, relu_r2s)
    else:
        t_stat, p_value = np.nan, np.nan
    
    # Spline advantage
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)
    
    print("\n" + "="*60)
    print("SPLINE vs RELU COMPARISON")
    print("="*60)
    
    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)
    
    print("\nReLU:")
    print(f"  Mean ± Std:  {relu_stats['mean']:.4f} ± {relu_stats['std']:.4f}")
    print(f"  95% CI:      [{relu_stats['ci_low']:.4f}, {relu_stats['ci_high']:.4f}]")
    
    print("\nSpline:")
    print(f"  Mean ± Std:  {spline_stats['mean']:.4f} ± {spline_stats['std']:.4f}")
    print(f"  95% CI:      [{spline_stats['ci_low']:.4f}, {spline_stats['ci_high']:.4f}]")
    
    print("\nSpline Advantage (%):")
    print(f"  Mean ± Std:  {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%")
    print(f"  Range:       [{adv_stats['min']:+.2f}%, {adv_stats['max']:+.2f}%]")
    print(f"  95% CI:      [{adv_stats['ci_low']:+.2f}%, {adv_stats['ci_high']:+.2f}%]")
    
    if not np.isnan(p_value):
        print(f"\nPaired t-test: t={t_stat:.3f}, p={p_value:.4f}")
        if p_value < 0.05:
            if adv_stats['mean'] > 0:
                print("  ✅ SIGNIFICANT: Spline wins (p < 0.05)")
            else:
                print("  ✅ SIGNIFICANT: ReLU wins (p < 0.05)")
        else:
            print("  ❌ NOT SIGNIFICANT: No clear winner (p ≥ 0.05)")
    
    # Practical significance
    if adv_stats['ci_low'] > 1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, advantage > 1%")
    elif adv_stats['ci_high'] < -1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, disadvantage > 1%")
    else:
        print("\n  ⚠️  NO PRACTICAL SIGNIFICANCE: 95% CI includes small effects")
    
    print("="*60)
    
    return adv_stats


print("✅ Statistics utilities loaded")

---
## Load Data

In [ ]:
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"
    data_path = 'etopo_60s.nc'
else:
    data_path = 'etopo_60s.nc'
    if not os.path.exists(data_path):
        print(f"❌ Data file not found: {data_path}")

ds = xr.open_dataset(data_path)
elevation = ds['z'].values
lats = ds['lat'].values
lons = ds['lon'].values

print(f"✅ Elevation data loaded: {elevation.shape}")
print(f"   Latitude range: [{lats.min():.2f}, {lats.max():.2f}]")
print(f"   Longitude range: [{lons.min():.2f}, {lons.max():.2f}]")
print(f"   Elevation range: [{elevation.min():.2f}, {elevation.max():.2f}] meters")
print("="*70)

---
## Experiment 1: Global Multi-Seed Validation

**Goal**: Validate NB19's +0.36% spline advantage is reproducible

**Config**: 15K samples, global, L=10, 100 epochs, 10 seeds

In [ ]:
print("="*80)
print("EXPERIMENT 1: GLOBAL MULTI-SEED VALIDATION (NB19 Reproducibility)")
print("="*80)

SEEDS = range(42, 52)  # 10 seeds: 42-51
ACTIVATIONS = ['relu', 'spline', 'siren']
N_SAMPLES = 15000
EPOCHS = 100

results_exp1 = []

for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"Seed {seed} ({seed-41}/10)")
    print("="*80)
    
    # Sample data
    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        elevation, lons, lats, n_samples=N_SAMPLES, seed=seed
    )
    print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")
    
    for act in ACTIVATIONS:
        print(f"\n  {act.upper()}...", end=" ")
        
        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_elevation_model(
            f'global_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            epochs=EPOCHS,
            verbose=False
        )
        
        res['seed'] = seed
        res['activation'] = act
        res['n_samples'] = N_SAMPLES
        
        results_exp1.append(res)
        print(f"R²={res['r2']:.4f}, Time={res['time']:.1f}s")

# Convert to DataFrame
df_exp1 = pd.DataFrame(results_exp1)

print("\n" + "="*80)
print("EXPERIMENT 1 SUMMARY: ALL RUNS")
print("="*80)
print(df_exp1[['seed', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

In [ ]:
# Statistical analysis
print("\n" + "="*80)
print("EXPERIMENT 1: STATISTICAL ANALYSIS")
print("="*80)

for act in ACTIVATIONS:
    act_data = df_exp1[df_exp1['activation'] == act]
    r2_values = act_data['r2'].values
    stats_dict = compute_stats(r2_values)
    print(f"\n{act.upper()}:")
    print_stats(f"  R² Statistics", stats_dict)

# ReLU vs Spline comparison
relu_r2s = df_exp1[df_exp1['activation'] == 'relu']['r2'].values
spline_r2s = df_exp1[df_exp1['activation'] == 'spline']['r2'].values

adv_stats = compare_activations(relu_r2s, spline_r2s)

# Compare to NB19
print("\n" + "="*80)
print("COMPARISON TO NB19 (Single Seed)")
print("="*80)
print("\nNB19 Results:")
print("  ReLU:   R² = 0.8997")
print("  Spline: R² = 0.9030")
print("  Advantage: +0.36%")
print("\nNB21 Results (10 seeds):")
print(f"  ReLU:   R² = {relu_r2s.mean():.4f} ± {relu_r2s.std():.4f}")
print(f"  Spline: R² = {spline_r2s.mean():.4f} ± {spline_r2s.std():.4f}")
print(f"  Advantage: {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%")

if adv_stats['cv'] < 50:  # CV < 50% considered stable
    print("\n✅ STABLE: Results reproducible (CV < 50%)")
else:
    print("\n❌ UNSTABLE: High variance (CV ≥ 50%)")

print("="*80)

---
## Experiment 2: Sample Size Sensitivity

**Goal**: Determine at what sample size results stabilize

**Config**: 5K, 10K, 20K, 50K samples, 10 seeds each

In [ ]:
print("="*80)
print("EXPERIMENT 2: SAMPLE SIZE SENSITIVITY")
print("="*80)

SAMPLE_SIZES = [5000, 10000, 20000, 50000]
SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline']  # Focus on main comparison

results_exp2 = []

for n_samples in SAMPLE_SIZES:
    print(f"\n{'='*80}")
    print(f"Sample Size: {n_samples:,}")
    print("="*80)
    
    for seed in SEEDS:
        print(f"\n  Seed {seed} ({seed-41}/10)", end=" ")
        
        coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
            elevation, lons, lats, n_samples=n_samples, seed=seed
        )
        
        for act in ACTIVATIONS:
            print(f"{act.upper()}", end=" ")
            
            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
            
            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )
            
            res = train_elevation_model(
                f'n{n_samples}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                epochs=EPOCHS,
                verbose=False
            )
            
            res['seed'] = seed
            res['activation'] = act
            res['n_samples'] = n_samples
            
            results_exp2.append(res)
            print(f"({res['r2']:.4f})", end=" ")
        
        print()  # Newline after seed

df_exp2 = pd.DataFrame(results_exp2)

print("\n" + "="*80)
print("EXPERIMENT 2 COMPLETE")
print("="*80)

In [ ]:
# Analyze variance vs sample size
print("\n" + "="*80)
print("EXPERIMENT 2: VARIANCE vs SAMPLE SIZE")
print("="*80)

print("\n{:>10s} {:>15s} {:>15s} {:>10s} {:>10s}".format(
    "N Samples", "ReLU R² (Mean)", "Spline R² (Mean)", "Adv (%)", "Adv CV (%)"
))
print("-" * 80)

for n_samples in SAMPLE_SIZES:
    subset = df_exp2[df_exp2['n_samples'] == n_samples]
    
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
    
    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)
    
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)
    
    print("{:>10,d} {:>15s} {:>15s} {:>10.2f} {:>10.2f}".format(
        n_samples,
        f"{relu_stats['mean']:.4f}±{relu_stats['std']:.4f}",
        f"{spline_stats['mean']:.4f}±{spline_stats['std']:.4f}",
        adv_stats['mean'],
        adv_stats['cv']
    ))

print("\n" + "="*80)
print("INTERPRETATION:")
print("  - Adv CV < 20%: Stable results")
print("  - Adv CV 20-50%: Moderate variance")
print("  - Adv CV > 50%: High variance, unreliable")
print("="*80)

---
## Experiment 3: Regional Multi-Seed (Larger Samples)

**Goal**: Test if terrain patterns stabilize with 10K-20K samples

**Regions**: Himalayas (mountain), Sahara (flat)

In [ ]:
print("="*80)
print("EXPERIMENT 3: REGIONAL MULTI-SEED WITH LARGER SAMPLES")
print("="*80)

REGIONS = {
    'asia_himalayas': {
        'lat_min': 25, 'lat_max': 40,
        'lon_min': 70, 'lon_max': 100,
        'terrain': 'mountainous',
    },
    'africa_sahara': {
        'lat_min': 15, 'lat_max': 30,
        'lon_min': -10, 'lon_max': 30,
        'terrain': 'flat',
    },
}

REGIONAL_SAMPLE_SIZES = [10000, 20000]
SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline']

results_exp3 = []

for region_name, region_bounds in REGIONS.items():
    print(f"\n{'='*80}")
    print(f"Region: {region_name.upper()} ({region_bounds['terrain']})")
    print("="*80)
    
    for n_samples in REGIONAL_SAMPLE_SIZES:
        print(f"\n  Sample Size: {n_samples:,}")
        print("  " + "-"*60)
        
        for seed in SEEDS:
            print(f"    Seed {seed}: ", end="")
            
            coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
                elevation, lons, lats, region_bounds, n_samples=n_samples, seed=seed
            )
            
            for act in ACTIVATIONS:
                kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
                
                enc = UniversalEncoder(
                    input_type='sh',
                    sh_legendre_polys=10,
                    activation_type=act,
                    activation_kwargs=kwargs
                )
                
                res = train_elevation_model(
                    f'{region_name}_n{n_samples}_seed{seed}_{act}',
                    enc,
                    coords_train, vals_train,
                    coords_test, vals_test,
                    epochs=EPOCHS,
                    verbose=False
                )
                
                res['seed'] = seed
                res['activation'] = act
                res['n_samples'] = n_samples
                res['region'] = region_name
                res['terrain'] = region_bounds['terrain']
                
                results_exp3.append(res)
                print(f"{act}={res['r2']:.4f} ", end="")
            
            print()  # Newline

df_exp3 = pd.DataFrame(results_exp3)

print("\n" + "="*80)
print("EXPERIMENT 3 COMPLETE")
print("="*80)

In [ ]:
# Analyze terrain effects with proper statistics
print("\n" + "="*80)
print("EXPERIMENT 3: TERRAIN EFFECT ANALYSIS")
print("="*80)

for region_name in REGIONS.keys():
    region_data = df_exp3[df_exp3['region'] == region_name]
    terrain = region_data['terrain'].iloc[0]
    
    print(f"\n{'='*80}")
    print(f"{region_name.upper()} ({terrain})")
    print("="*80)
    
    for n_samples in REGIONAL_SAMPLE_SIZES:
        subset = region_data[region_data['n_samples'] == n_samples]
        
        relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
        spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
        
        print(f"\n--- N = {n_samples:,} ---")
        compare_activations(relu_r2s, spline_r2s)

# Compare terrains
print("\n" + "="*80)
print("TERRAIN COMPARISON (20K samples)")
print("="*80)

mountain_data = df_exp3[(df_exp3['terrain'] == 'mountainous') & (df_exp3['n_samples'] == 20000)]
flat_data = df_exp3[(df_exp3['terrain'] == 'flat') & (df_exp3['n_samples'] == 20000)]

mountain_relu = mountain_data[mountain_data['activation'] == 'relu']['r2'].values
mountain_spline = mountain_data[mountain_data['activation'] == 'spline']['r2'].values
mountain_adv = 100 * (mountain_spline - mountain_relu) / mountain_relu

flat_relu = flat_data[flat_data['activation'] == 'relu']['r2'].values
flat_spline = flat_data[flat_data['activation'] == 'spline']['r2'].values
flat_adv = 100 * (flat_spline - flat_relu) / flat_relu

mountain_stats = compute_stats(mountain_adv)
flat_stats = compute_stats(flat_adv)

print("\nMountainous (Himalayas):")
print(f"  Spline Advantage: {mountain_stats['mean']:+.2f}% ± {mountain_stats['std']:.2f}%")
print(f"  95% CI: [{mountain_stats['ci_low']:+.2f}%, {mountain_stats['ci_high']:+.2f}%]")

print("\nFlat (Sahara):")
print(f"  Spline Advantage: {flat_stats['mean']:+.2f}% ± {flat_stats['std']:.2f}%")
print(f"  95% CI: [{flat_stats['ci_low']:+.2f}%, {flat_stats['ci_high']:+.2f}%]")

# Test if terrains differ
t_stat, p_value = stats.ttest_ind(mountain_adv, flat_adv)
print(f"\nTerrain Difference Test: t={t_stat:.3f}, p={p_value:.4f}")
if p_value < 0.05:
    print("  ✅ SIGNIFICANT: Terrain matters (p < 0.05)")
else:
    print("  ❌ NOT SIGNIFICANT: No terrain effect (p ≥ 0.05)")

print("="*80)

---
## Experiment 4: Extended Training (Convergence Analysis)

**Goal**: Check if 100 epochs is sufficient or models need more training

**Config**: 200 epochs, track R² every epoch, 5 seeds

In [ ]:
print("="*80)
print("EXPERIMENT 4: CONVERGENCE ANALYSIS (200 Epochs)")
print("="*80)

SEEDS = range(42, 47)  # 5 seeds
ACTIVATIONS = ['relu', 'spline']
EPOCHS_CONV = 200

results_exp4 = []

for seed in SEEDS:
    print(f"\nSeed {seed}:")
    
    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        elevation, lons, lats, n_samples=15000, seed=seed
    )
    
    for act in ACTIVATIONS:
        print(f"  {act.upper()}...", end=" ")
        
        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_elevation_model(
            f'conv_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            epochs=EPOCHS_CONV,
            verbose=False,
            track_epochs=True
        )
        
        res['seed'] = seed
        res['activation'] = act
        
        results_exp4.append(res)
        print(f"Best R²={res['r2']:.4f}")

print("\n" + "="*80)
print("CONVERGENCE SUMMARY")
print("="*80)

# Check if models improved after epoch 100
for act in ACTIVATIONS:
    act_results = [r for r in results_exp4 if r['activation'] == act]
    
    r2_at_100 = []
    r2_at_200 = []
    
    for res in act_results:
        history = res['r2_history']
        r2_at_100.append(max(history[:100]))  # Best in first 100
        r2_at_200.append(max(history))  # Best overall
    
    improvement = np.array(r2_at_200) - np.array(r2_at_100)
    imp_stats = compute_stats(improvement)
    
    print(f"\n{act.upper()}:")
    print(f"  R² at epoch 100: {np.mean(r2_at_100):.4f} ± {np.std(r2_at_100):.4f}")
    print(f"  R² at epoch 200: {np.mean(r2_at_200):.4f} ± {np.std(r2_at_200):.4f}")
    print(f"  Improvement:     {imp_stats['mean']:.4f} ± {imp_stats['std']:.4f}")
    
    if imp_stats['mean'] > 0.01:  # More than 1% absolute improvement
        print("  ⚠️  UNDERTRAINING: Models still improving after 100 epochs")
    else:
        print("  ✅ CONVERGED: 100 epochs sufficient")

print("="*80)

---
## Experiment 5: Spatial Blocking Sensitivity

**Goal**: Test how block size affects variance

**Config**: 1°, 2°, 5°, 10° grids, 5 seeds each

In [ ]:
print("="*80)
print("EXPERIMENT 5: SPATIAL BLOCKING SENSITIVITY")
print("="*80)

GRID_SIZES = [1.0, 2.0, 5.0, 10.0]  # degrees
SEEDS = range(42, 47)  # 5 seeds
ACTIVATIONS = ['relu', 'spline']

results_exp5 = []

for grid_size in GRID_SIZES:
    print(f"\n{'='*80}")
    print(f"Grid Size: {grid_size}° (~{int(grid_size * 111)} km at equator)")
    print(f"Approx blocks: {int(360/grid_size) * int(180/grid_size)}")
    print("="*80)
    
    for seed in SEEDS:
        print(f"  Seed {seed}: ", end="")
        
        coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
            elevation, lons, lats, n_samples=15000, grid_size=grid_size, seed=seed
        )
        
        for act in ACTIVATIONS:
            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
            
            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )
            
            res = train_elevation_model(
                f'grid{grid_size}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                epochs=EPOCHS,
                verbose=False
            )
            
            res['seed'] = seed
            res['activation'] = act
            res['grid_size'] = grid_size
            
            results_exp5.append(res)
            print(f"{act}={res['r2']:.4f} ", end="")
        
        print()

df_exp5 = pd.DataFrame(results_exp5)

print("\n" + "="*80)
print("SPATIAL BLOCKING ANALYSIS")
print("="*80)

print("\n{:>10s} {:>12s} {:>15s} {:>10s}".format(
    "Grid (°)", "# Blocks", "Spline Adv (%)", "Adv CV (%)"
))
print("-" * 60)

for grid_size in GRID_SIZES:
    subset = df_exp5[df_exp5['grid_size'] == grid_size]
    
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
    
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)
    
    n_blocks = int(360/grid_size) * int(180/grid_size)
    
    print("{:>10.1f} {:>12d} {:>15s} {:>10.2f}".format(
        grid_size,
        n_blocks,
        f"{adv_stats['mean']:+.2f}±{adv_stats['std']:.2f}",
        adv_stats['cv']
    ))

print("\n" + "="*80)
print("INTERPRETATION:")
print("  - Smaller grids (more blocks) should reduce variance")
print("  - If CV remains high regardless: intrinsic variance, not blocking")
print("="*80)

---
## Final Summary

Synthesize all experiments

In [ ]:
print("="*80)
print("FINAL SUMMARY: REPRODUCIBILITY VALIDATION")
print("="*80)

print("\n" + "="*80)
print("EXPERIMENT 1: Global Multi-Seed (NB19 Validation)")
print("="*80)
relu_r2s = df_exp1[df_exp1['activation'] == 'relu']['r2'].values
spline_r2s = df_exp1[df_exp1['activation'] == 'spline']['r2'].values
advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
adv_stats_exp1 = compute_stats(advantages)

print(f"Spline Advantage: {adv_stats_exp1['mean']:+.2f}% ± {adv_stats_exp1['std']:.2f}%")
print(f"95% CI: [{adv_stats_exp1['ci_low']:+.2f}%, {adv_stats_exp1['ci_high']:+.2f}%]")
print(f"CV: {adv_stats_exp1['cv']:.1f}%")

if adv_stats_exp1['cv'] < 20:
    print("\n✅ REPRODUCIBLE: Low variance (CV < 20%)")
elif adv_stats_exp1['cv'] < 50:
    print("\n⚠️  MODERATE VARIANCE: CV 20-50%")
else:
    print("\n❌ HIGH VARIANCE: CV > 50%, results unreliable")

print("\n" + "="*80)
print("EXPERIMENT 2: Sample Size Effect")
print("="*80)

for n_samples in SAMPLE_SIZES:
    subset = df_exp2[df_exp2['n_samples'] == n_samples]
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)
    
    print(f"N={n_samples:>6,}: Advantage = {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%, CV = {adv_stats['cv']:.1f}%")

print("\n" + "="*80)
print("EXPERIMENT 3: Terrain Effects (20K samples)")
print("="*80)

for region_name in REGIONS.keys():
    subset = df_exp3[(df_exp3['region'] == region_name) & (df_exp3['n_samples'] == 20000)]
    terrain = subset['terrain'].iloc[0]
    
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)
    
    print(f"{region_name:20s} ({terrain:12s}): {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%, CV = {adv_stats['cv']:.1f}%")

print("\n" + "="*80)
print("KEY TAKEAWAYS")
print("="*80)

print("\n1. REPRODUCIBILITY:")
if adv_stats_exp1['cv'] < 20:
    print("   ✅ Global results are reproducible with multi-seed validation")
else:
    print("   ❌ Even global results show high variance - fundamental issue")

print("\n2. SAMPLE SIZE:")
print("   See Exp 2 results above - check if variance decreases with N")

print("\n3. TERRAIN EFFECTS:")
print("   See Exp 3 results above - check if patterns are consistent")

print("\n4. CONVERGENCE:")
print("   See Exp 4 - check if 100 epochs is sufficient")

print("\n5. SPATIAL BLOCKING:")
print("   See Exp 5 - check if block size matters")

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

if adv_stats_exp1['cv'] < 20 and adv_stats_exp1['ci_low'] > 1.0:
    print("\n✅ PROCEED: Results stable, spline shows advantage")
    print("   Continue with Exp 3-5 in NB20 using larger samples")
elif adv_stats_exp1['cv'] < 20 and adv_stats_exp1['ci_high'] < -1.0:
    print("\n✅ PROCEED: Results stable, ReLU wins consistently")
    print("   Write up as negative result: 'SH + ReLU Optimal'")
elif adv_stats_exp1['cv'] < 20:
    print("\n⚠️  INCONCLUSIVE: Results stable but effect size small")
    print("   May not be practically significant even if statistically so")
else:
    print("\n❌ STOP: High variance even with multi-seed validation")
    print("   Focus on understanding why variance is high (NB22: Mechanistic Analysis)")

print("\n" + "="*80)

---
## Optional: Save to Google Drive

If running on Colab, can save results for persistence

In [ ]:
# Optional: Mount Google Drive and save results
if 'COLAB_GPU' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    
    save_dir = '/content/drive/MyDrive/learned_activation_results/nb21/'
    os.makedirs(save_dir, exist_ok=True)
    
    df_exp1.to_csv(f'{save_dir}exp1_global_multiseed.csv', index=False)
    df_exp2.to_csv(f'{save_dir}exp2_sample_size.csv', index=False)
    df_exp3.to_csv(f'{save_dir}exp3_regional_multiseed.csv', index=False)
    df_exp5.to_csv(f'{save_dir}exp5_spatial_blocking.csv', index=False)
    
    print(f"✅ Results saved to Google Drive: {save_dir}")
else:
    print("Not running on Colab, skip Drive save")